## Anomaly detection

Anomaly detection in helicopter vibration data:
- https://www.kaggle.com/datasets/nagasai524/anomaly-detection-in-helicopter


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import multiprocessing
from time import perf_counter
from datetime import datetime

nr_of_cpus = multiprocessing.cpu_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

source = "data"
output_folder = "trained_lstm_models"

torch.backends.cudnn.benchmark = True

print(f"There are {nr_of_cpus} CPU cores and " + "Cuda is available." if device.type == "cuda" else "Cuda is not available.")


### Load the prepared datasets
The test and training datasets are loaded using more efficient file format. We'll keep using csv for the labels because it's not a large file.

In [ ]:
# Load the dataset

train_data_file = f"{source}\\helicopter_train.npy"
test_data_file = f"{source}\\helicopter_test.npy"
labels_path = f"{source}\\dfvalid_groundtruth.csv"

labels = torch.from_numpy(pd.read_csv(labels_path)["anomaly"].values).float()

train_data = torch.from_numpy(np.load(train_data_file)).float().unsqueeze(-1)
test_data = torch.from_numpy(np.load(test_data_file)).float().unsqueeze(-1)

print(f"There are {len(train_data)} training samples and {len(test_data)} test samples.")

### Define LSTM-based AutoEncoder

Two different models will be created using this class.

In [ ]:

class LSTMAutoencoder(nn.Module):

    def __init__(self, hidden_size, nr_of_layers= 1):
        super().__init__()
        self.name = f"{self.__class__.__name__}_{datetime.now().strftime('%m%d%H%M')}"

        self.nr_of_layers = nr_of_layers
        self.hidden_size = hidden_size

        self.encoder = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=nr_of_layers, batch_first=True)
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=nr_of_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, 1)

    def __str__(self):
        return self.name

    def forward(self, x):
        # Encode
        _, (h_n, c_n) = self.encoder(x)

        seq_len = x.size(1)

        z = h_n[-1]                 # This is the last layer of the encoder, i.e the encoded sequence
        z_seq = z.unsqueeze(1).repeat(1, seq_len, 1)
    
        # Decode
        reconstructed, _ = self.decoder(z_seq, (h_n, c_n))

        return self.output_layer(reconstructed)

class LSTMVariationalAE(nn.Module):
    def __init__(self, hidden_size, latent_dim, nr_of_layers: int = 1):
        super().__init__()
        self.name = f"{self.__class__.__name__}_{datetime.now().strftime('%m%d%H%M')}"

        self.hidden_size = hidden_size
        self.nr_of_layers = nr_of_layers
        self.latent_dim = latent_dim

        self.encoder = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=nr_of_layers, batch_first=True)

        self.mean_branch   = nn.Linear(hidden_size, latent_dim)
        self.logvar_branch = nn.Linear(hidden_size, latent_dim)

        self.latent_to_hidden = nn.Linear(latent_dim, hidden_size)

        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=nr_of_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, 1)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def __str__(self):
        return self.name

    def forward(self, x: torch.Tensor):

        _, (h_n, _) = self.encoder(x)           # batch * seq_len * input_size
        encoded = h_n[-1]                       # batch * hidden_size

        mu = self.mean_branch(encoded)          # batch * latent_dim
        logvar = self.logvar_branch(encoded)    # batch * latent_dim
        z = self.reparameterize(mu, logvar)
        seq_len = x.size(1)

        z_seq = self.latent_to_hidden(z).unsqueeze(1).repeat(1, seq_len, 1)    # batch * seq_len * latent_dim
        reconstructed, _ = self.decoder(z_seq)

        return self.output_layer(reconstructed), mu, logvar


In [ ]:
# Create models and send them to GPU

ae_model = LSTMAutoencoder(hidden_size=128).to(device)
vae_model = LSTMVariationalAE(hidden_size=128, latent_dim=16).to(device)

### Early stopping
This function is used to detect overtraining

In [ ]:
def should_stop(losses, patience=5, percent_of_improvement=0.01):
    """Returns True if loss hasn't improved a certain percentage in a number of epochs."""
    if len(losses) <= patience:
        return False

    best_loss = min(losses[:-patience])
    threshold = best_loss * (1 - percent_of_improvement)

    return min(losses[-patience:]) >= threshold

### Train first model

In [ ]:
# Move training data to GPU
train_data = train_data.to(device);

In [ ]:

nr_of_epochs = 20
batch_size = 8
nr_of_batches = len(train_data) // batch_size

optimizer_ae = optim.Adam(ae_model.parameters(), lr=0.001)
recon_criterion = nn.MSELoss(reduction='mean')

lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer_ae,
                                                        max_lr=0.01,
                                                        steps_per_epoch=nr_of_batches,
                                                        epochs=nr_of_epochs,
                                                        pct_start=0.25,
                                                        anneal_strategy='cos',
                                                        div_factor=25.0,
                                                        final_div_factor=1000.0,
                                                        cycle_momentum=False)

loss_history = []
scaler = GradScaler()              # for AMP
epoch_durations = []
training_log = ""

for epoch in range(1, nr_of_epochs + 1):
    start = perf_counter()
    ae_model.train()
    epoch_loss = 0.0

    for i in range(0, len(train_data), batch_size):
        seq = train_data[i:i+batch_size]

        optimizer_ae.zero_grad(set_to_none=True)

        with autocast():
            recon_seq = ae_model(seq)
            rec_error = recon_criterion(recon_seq, seq)

        scaler.scale(rec_error).backward()

        scaler.unscale_(optimizer_ae)
        nn.utils.clip_grad_norm_(ae_model.parameters(), max_norm=1.0)

        scaler.step(optimizer_ae)
        scaler.update()

        lr_scheduler.step()

        epoch_loss += rec_error.item()

    avg_loss = epoch_loss / nr_of_batches
    loss_history.append(avg_loss)

    duration = perf_counter() - start
    epoch_durations.append(duration)
    time_left = (nr_of_epochs - epoch) * (sum(epoch_durations) / len(epoch_durations))

    log_str = (f"[AE] Epoch {epoch}/{nr_of_epochs}  "
               f"Rec Loss: {avg_loss:.6f}  Duration: {duration:.1f}s  "
               f"Time left: {time_left:.1f}s")
    training_log += log_str + "\n"
    print(log_str)

    if should_stop(loss_history):
        early_stopp_str = f"Early stopping at {epoch}"
        log_str += "\n" + early_stopp_str
        print(early_stopp_str)
        break

In [ ]:
try:
    torch.save(ae_model.state_dict(), f"{output_folder}\\{ae_model}.pth")
    with open("log.txt", "a") as file:
        file.write(training_log + "\n")
except Exception as e:
    print(f"Could not save AE model due to {e}")

In [ ]:

batch_size = 8
nr_of_epochs = 20
nr_of_batches = len(train_data) // batch_size

optimizer_vae = optim.Adam(vae_model.parameters(), lr=0.001)
scaler = GradScaler()
recon_criterion = nn.MSELoss(reduction='mean')
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer_vae,
                                                    max_lr=0.01,
                                                        steps_per_epoch=nr_of_batches,
                                                        epochs=nr_of_epochs,
                                                        pct_start=0.25,
                                                        anneal_strategy='cos',
                                                        div_factor=25.0,
                                                        final_div_factor=1000.0,
                                                        cycle_momentum=False)

loss_history = []
epoch_durations = []
training_log = ""
beta = 1.0

for epoch in range(1, nr_of_epochs + 1):
    start = perf_counter()
    vae_model.train()
    epoch_loss = 0.0

    for i in range(0, len(train_data), batch_size):
        seq = train_data[i:i+batch_size]

        optimizer_vae.zero_grad(set_to_none=True)

        with autocast():
            recon_seq, mu, logvar = vae_model(seq)
            rec_error = recon_criterion(recon_seq, seq)
            kl_error = -0.5 * torch.mean((1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=-1))
            loss = rec_error + beta * kl_error

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer_vae)
        nn.utils.clip_grad_norm_(vae_model.parameters(), max_norm=1.0)

        scaler.step(optimizer_vae)
        scaler.update()

        lr_scheduler.step()

        epoch_loss += loss.item()

        break

    avg_loss = epoch_loss / nr_of_batches
    loss_history.append(avg_loss)
    duration = perf_counter() - start
    epoch_durations.append(duration)
    time_left = (nr_of_epochs - epoch) * (sum(epoch_durations) / len(epoch_durations))

    log_str = (
        f"[VAE] Epoch {epoch}/{nr_of_epochs}  "
        f"Loss: {avg_loss:.6f} "
        f"(Rec: {rec_error.item():.6f}, KL: {kl_error.item():.6f})  "
        f"Duration: {duration:.1f}s  Time left: {time_left:.1f}s"
    )
    training_log += log_str + "\n"
    print(log_str)

    if should_stop(loss_history):
        early_stopp_str = f"Early stopping at {epoch}"
        log_str += "\n" + early_stopp_str
        print(early_stopp_str)
        break


In [ ]:
try:
    torch.save(vae_model.state_dict(), f"{output_folder}\\{vae_model}.pth")
    with open("log.txt", "a") as file:
        file.write(training_log + "\n")
except Exception as e:
    print(f"Could not save VAE model due to {e}")

In [ ]:
# Free up memory on GPU
torch.cuda.empty_cache()

ae_model.to(device)
vae_model.to(device)

test_data = test_data.to(device);

### Evaluate models

In [ ]:
test_batch_size = 8

ae_errors = list()
vae_errors = list()

for model, name in zip([ae_model, vae_model], ["AutoEncoder", "Variational AutoEncoder"]):
    model.eval()

    with torch.no_grad():
        for i in range(0, len(test_data), test_batch_size):
            original = test_data[i:i+test_batch_size]

            if name == "AutoEncoder":
                reconstructed = model(original)
                ae_errors.append((reconstructed - original).flatten(1))
            else:
                reconstructed, *_ = model(original)
                vae_errors.append((reconstructed - original).flatten(1))


In [ ]:
ae_threshold = 0.67
vae_threshold = 0.75

for model, name, raw_errors, threshold in zip([ae_model, vae_model], ["AutoEncoder", "Variational AutoEncoder"], [ae_errors, vae_errors], [ae_threshold, vae_threshold]):

    errors = torch.cat(raw_errors).cpu()

    mae = errors.abs().mean()
    mse = errors.pow(2).mean(1)         # Per sample MSE

    predictions = (mse > threshold).int().cpu()

    roc_auc = roc_auc_score(labels, mse)                # ROC AUC expects probabilities
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    mse = errors.pow(2).mean()      # Model MSE

    print(f"{name} Evaluation\n{'-' * 24}\n* MAE: {mae:.4f}\n* MSE: {mse:.4f}\n* ROC-AUC: {roc_auc:.3f}\n{'-' * 24}\n* Accuracy: {accuracy:.4f}\n* Precision: {precision:.4f}\n* Recall: {recall:.4f}\n* F1: {f1:.4f}\n{'-' * 24}\n")